
# ReliableVAEP PCS/RAR
## Fully standalone Kaggle notebook

This notebook performs the **static reliability-adjusted ranking experiment** from start to finish.

It does **not** require:

- `main_bootstrap_vaep90.npy`
- `06_main_deterministic_leaderboard.csv`
- `premier_league_2015_16_player_match.csv`
- another ReliableVAEP notebook to be run first
- manually uploaded intermediate data

It downloads StatsBomb open data, constructs canonical SPADL actions, fits/cross-fits the canonical VAEP probability models, calculates action VAEP, aggregates player-match VAEP, bootstraps player rankings, fits the Pairwise Consensus Score (PCS), produces Reliability-Adjusted Rank (RAR), and validates RAR against raw VAEP using independent non-overlapping match samples.

## Datasets

### Primary
Barcelona, La Liga, 2015/16–2019/20

### Temporal validation
Barcelona, La Liga, 2008/09–2012/13

The match sets are explicitly checked to be non-overlapping.

## Four analyses

1. Primary outfield-only cohort, **main analysis**
2. Primary all-player cohort, sensitivity
3. Validation outfield-only cohort, **main replication**
4. Validation all-player cohort, sensitivity

## What the notebook tests

The full-data output is:

\[
\text{VAEP/90}
\rightarrow
\text{bootstrap pairwise probabilities}
\rightarrow
\text{PCS}
\rightarrow
\text{RAR}
\]

For every pair:

\[
P(V_i > V_j)
\]

is estimated from match-level bootstrap draws.

A Bradley-Terry-style latent model then fits:

\[
P(V_i>V_j)
\approx
\frac{1}{1+\exp[-(S_i-S_j)]}
\]

where \(S_i\) is PCS. Sorting PCS gives one static RAR ranking.

## Critical validation

RAR is **not** automatically considered better.

At every tested match count \(n\):

1. create two non-overlapping \(n\)-match samples per player;
2. calculate raw VAEP ranking independently in A and B;
3. bootstrap A and B independently;
4. fit PCS/RAR independently;
5. calculate:

\[
\rho_{\mathrm{raw}}
=
\mathrm{Spearman}(R^{VAEP}_A,R^{VAEP}_B)
\]

and

\[
\rho_{\mathrm{RAR}}
=
\mathrm{Spearman}(R^{RAR}_A,R^{RAR}_B)
\]

6. evaluate:

\[
\Delta\rho =
\rho_{\mathrm{RAR}}-\rho_{\mathrm{raw}}
\]

Only consistent positive \(\Delta\rho\) across primary and validation analyses would support calling RAR a **more reproducible reliability-adjusted ranking**.

## Kaggle instructions

Before running:

1. Open Kaggle **Settings**
2. Turn **Internet = On**
3. Run all cells from the top

The notebook creates an isolated environment under:

`/kaggle/working/rv_vaep_env`

and does not downgrade Kaggle's own NumPy/pandas stack.


> **Canonical ReliableVAEP bootstrap setting for the final SSAC analysis:** `FULL_BOOTSTRAP_B = 3500`.
>
> The value is not an arbitrary round number. ReliableVAEP treats a pairwise ordering as supported at
> \(P_{ij}\ge 0.90\) (or \(\le 0.10\)). Requiring a 95% Wilson Monte Carlo half-width of at most
> 0.01 at \(p=0.90\) gives a minimum of 3,461 resamples. The final analyses therefore use 3,500
> player-preserving match-level bootstrap draws, the smallest 100-draw increment above that bound.
> The completed cross-seed sensitivity test confirmed very high rank-order and pair-classification
> stability at \(B=3500\) across the study datasets.
>
> **Important:** `INNER_BOOTSTRAP_B` values used inside repeated split-sample validation are left at
> their original values. They are nested computational approximations for hundreds of repeated
> validation runs and are not used to produce the paper's primary full-data rank intervals or
> 0.90 pairwise-support decisions.

In [1]:
# FINAL SSAC RUN CONFIGURATION CHECK
CANONICAL_FULL_BOOTSTRAP_B = 3500
print("ReliableVAEP final SSAC run")
print("Canonical full-data bootstrap draws:", CANONICAL_FULL_BOOTSTRAP_B)
print("Run this notebook from top to bottom and use only the newly generated outputs.")


ReliableVAEP final SSAC run
Canonical full-data bootstrap draws: 3500
Run this notebook from top to bottom and use only the newly generated outputs.


In [2]:

# ================================================================
# CELL 1 — KAGGLE-SAFE ISOLATED ENVIRONMENT
# ================================================================
#
# IMPORTANT:
# Kaggle Internet must be ON.
#
# This setup:
#   1. uses /kaggle/working only;
#   2. installs only the small pure-Python "virtualenv" helper into
#      Kaggle's host environment if it is missing;
#   3. creates /kaggle/working/rv_vaep_env;
#   4. installs the complete VAEP stack INSIDE that environment;
#   5. leaves Kaggle's NumPy/pandas/scipy/sklearn untouched.
#
# No runtime restart is required.
# ================================================================

import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

if not Path("/kaggle/working").exists():
    raise RuntimeError(
        "This notebook is the Kaggle-specific version, but /kaggle/working "
        "was not found. Please run it inside a Kaggle Notebook."
    )

if sys.version_info >= (3, 13):
    raise RuntimeError(
        "Please use a Kaggle Python runtime <= 3.12. "
        "socceraction 1.5.3 is not designed for Python 3.13."
    )

BASE_DIR = Path("/kaggle/working")
ENV_DIR = BASE_DIR / "rv_vaep_env"
ENV_PY = ENV_DIR / "bin" / "python"
ENV_PIP = ENV_DIR / "bin" / "pip"
READY = ENV_DIR / ".reliablevaep_kaggle_ready_v1"

IMPORT_TEST = r"""
import numpy
import pandas
import scipy
import sklearn
import xgboost
import socceraction

from socceraction.data.statsbomb import StatsBombLoader
from socceraction.vaep import VAEP
import socceraction.spadl as spadl

print("numpy        =", numpy.__version__)
print("pandas       =", pandas.__version__)
print("scipy        =", scipy.__version__)
print("sklearn      =", sklearn.__version__)
print("xgboost      =", xgboost.__version__)
print("socceraction =", getattr(socceraction, "__version__", "installed"))

# NumPy/pandas binary compatibility smoke test.
df = pandas.DataFrame({"x": numpy.arange(5, dtype=float)})
assert float(df["x"].sum()) == 10.0

print("KAGGLE_ISOLATED_ENV_OK")
"""


def run_capture(args, timeout=None):
    return subprocess.run(
        [str(x) for x in args],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )


def environment_is_healthy():
    if not (
        ENV_PY.exists()
        and ENV_PIP.exists()
        and READY.exists()
    ):
        return False

    test = run_capture(
        [ENV_PY, "-c", IMPORT_TEST]
    )

    if (
        test.returncode == 0
        and "KAGGLE_ISOLATED_ENV_OK" in test.stdout
    ):
        print("Existing isolated environment verified:")
        print(test.stdout)
        return True

    print("Existing isolated environment is incomplete/unhealthy.")
    print(test.stdout)
    return False


if not environment_is_healthy():

    # Remove only our own previous environment.
    if ENV_DIR.exists():
        print("Removing incomplete ReliableVAEP environment:", ENV_DIR)
        shutil.rmtree(ENV_DIR, ignore_errors=True)

    # ------------------------------------------------------------
    # Install virtualenv only if unavailable.
    # virtualenv is pure Python and does not alter NumPy/pandas.
    # ------------------------------------------------------------
    if importlib.util.find_spec("virtualenv") is None:
        print("Installing the lightweight virtualenv helper into Kaggle...")
        helper = run_capture([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-cache-dir",
            "virtualenv==20.28.0",
        ])

        print(helper.stdout)

        if helper.returncode != 0:
            raise RuntimeError(
                "Could not install virtualenv.\n"
                "Make sure Kaggle Internet is ON.\n\n"
                + helper.stdout
            )
    else:
        print("virtualenv helper is already available.")

    # ------------------------------------------------------------
    # Create isolated environment using virtualenv.
    # This avoids system ensurepip entirely.
    # ------------------------------------------------------------
    print("\nCreating isolated environment:", ENV_DIR)

    create = run_capture([
        sys.executable,
        "-m",
        "virtualenv",
        "--clear",
        str(ENV_DIR),
    ])

    print(create.stdout)

    if (
        create.returncode != 0
        or not ENV_PY.exists()
        or not ENV_PIP.exists()
    ):
        raise RuntimeError(
            "virtualenv could not create the isolated environment.\n\n"
            + create.stdout
        )

    # ------------------------------------------------------------
    # Install one internally consistent stack.
    # These packages exist ONLY inside /kaggle/working/rv_vaep_env.
    # ------------------------------------------------------------
    packages = [
        "numpy==1.26.4",
        "pandas==2.1.4",
        "scipy==1.13.1",
        "scikit-learn==1.5.2",
        "xgboost==2.1.4",
        "matplotlib==3.9.2",
        "tqdm==4.66.5",
        "multimethod==1.10",
        "pandera==0.17.2",
        "socceraction[statsbomb]==1.5.3",
    ]

    print("\nInstalling the isolated ReliableVAEP stack...")
    print("This can take a few minutes on the first run.")

    install = run_capture([
        ENV_PY,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        *packages,
    ])

    print(install.stdout)

    if install.returncode != 0:
        raise RuntimeError(
            "Package installation failed INSIDE the isolated environment.\n"
            "Kaggle's host NumPy/pandas environment has not been changed.\n\n"
            + install.stdout
        )

    # ------------------------------------------------------------
    # Verify dependency consistency.
    # ------------------------------------------------------------
    print("\nChecking isolated dependency consistency...")

    check = run_capture([
        ENV_PY,
        "-m",
        "pip",
        "check",
    ])

    print(
        check.stdout
        if check.stdout.strip()
        else "pip check: no dependency conflicts"
    )

    if check.returncode != 0:
        raise RuntimeError(
            "Dependency conflicts remain inside the isolated environment.\n\n"
            + check.stdout
        )

    # ------------------------------------------------------------
    # Import and binary compatibility test.
    # ------------------------------------------------------------
    print("\nVerifying isolated imports...")

    verify = run_capture([
        ENV_PY,
        "-c",
        IMPORT_TEST,
    ])

    print(verify.stdout)

    if (
        verify.returncode != 0
        or "KAGGLE_ISOLATED_ENV_OK" not in verify.stdout
    ):
        raise RuntimeError(
            "The isolated environment was created, but its scientific stack "
            "failed verification.\n\n"
            + verify.stdout
        )

    READY.write_text("ready", encoding="utf-8")

    print("\nKaggle isolated environment setup completed successfully.")

else:
    print("\nReusing the verified Kaggle isolated environment.")

print("\nAnalysis Python:", ENV_PY)
print("Kaggle host scientific packages were NOT downgraded or replaced.")


Installing the lightweight virtualenv helper into Kaggle...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 193.5 MB/s eta 0:00:00


Creating isolated environment: /kaggle/working/rv_vaep_env
created virtual environment CPython3.12.13.final.0-64 in 388ms
  creator CPython3Posix(dest=/kaggle/working/rv_vaep_env, clear=True, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.local/share/virtualenv)
    added seed packages: pip==24.3.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator


Installing the isolated ReliableVAEP stack...
This can take a few minutes on the first run.
Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
INFO: pip is looking at multiple versions of contourpy to determine which version is compatible with other 

In [3]:

# ================================================================
# CELL 2 — WRITE THE COMPLETE STANDALONE PCS/RAR ANALYSIS
# ================================================================

from pathlib import Path

BASE_DIR = Path("/kaggle/working")
SCRIPT_PATH = BASE_DIR / "reliablevaep_pcs_rar_standalone.py"

ANALYSIS_SCRIPT = '\n# ================================================================\n# ReliableVAEP Dual-Period Canonical Validation\n# Runs inside the isolated virtual environment.\n# ================================================================\n\nimport json\nimport os\nimport shutil\nimport time\nimport urllib.request\nimport warnings\nfrom pathlib import Path\n\n# Kaggle exports MPLBACKEND=module://matplotlib_inline.backend_inline.\n# The isolated environment should never inherit that notebook-only backend.\nos.environ["MPLBACKEND"] = "Agg"\n\nimport matplotlib\nmatplotlib.use("Agg", force=True)\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport xgboost as xgb\n\nfrom scipy.special import logit, expit\nfrom scipy.optimize import minimize\nfrom scipy.stats import spearmanr, wilcoxon\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score\nfrom sklearn.model_selection import GroupKFold, GroupShuffleSplit\nfrom tqdm.auto import tqdm\n\nfrom socceraction.data.statsbomb import StatsBombLoader\nimport socceraction.spadl as spadl\nfrom socceraction.vaep import VAEP\nfrom socceraction.vaep import formula as vaep_formula\n\nwarnings.filterwarnings("ignore")\n\nSEED = 20260903\nnp.random.seed(SEED)\n\nSTABILITY_TARGET = 0.80\nPAIRWISE_THRESHOLD = 0.90\n\n# Main run settings.\nREPEATS = 300\nPAIRWISE_REPEATS = 300\nPRECISION_REPEATS = 300\n\nOUTER_FOLDS = 5\nCALIBRATION_MATCH_FRACTION = 0.20\n\nPRIMARY_SEASONS = {\n    "2015/16": {"competition_id": 11, "season_id": 27},\n    "2016/17": {"competition_id": 11, "season_id": 2},\n    "2017/18": {"competition_id": 11, "season_id": 1},\n    "2018/19": {"competition_id": 11, "season_id": 4},\n    "2019/20": {"competition_id": 11, "season_id": 42},\n}\n\nVALIDATION_SEASONS = {\n    "2008/09": {"competition_id": 11, "season_id": 41},\n    "2009/10": {"competition_id": 11, "season_id": 21},\n    "2010/11": {"competition_id": 11, "season_id": 22},\n    "2011/12": {"competition_id": 11, "season_id": 23},\n    "2012/13": {"competition_id": 11, "season_id": 24},\n}\n\nROOT = Path("/kaggle/working")\nCACHE_ROOT = ROOT / "rv_dual_period_canonical_cache"\nOUT_ROOT = ROOT / "rv_dual_period_canonical_outputs"\nRAR_OUT_ROOT = ROOT / "rv_pcs_rar_outputs"\nCACHE_ROOT.mkdir(parents=True, exist_ok=True)\nOUT_ROOT.mkdir(parents=True, exist_ok=True)\nRAR_OUT_ROOT.mkdir(parents=True, exist_ok=True)\n\nBASE = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"\n\nprint("=" * 88)\nprint("ReliableVAEP canonical dual-period validation")\nprint("=" * 88)\nprint("Primary:", ", ".join(PRIMARY_SEASONS))\nprint("Validation:", ", ".join(VALIDATION_SEASONS))\nprint("Outputs:", OUT_ROOT)\nprint()\n\n\ndef get_json(url, path, retries=5):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n\n    if path.exists():\n        try:\n            with open(path, encoding="utf-8") as f:\n                return json.load(f)\n        except Exception:\n            path.unlink()\n\n    for attempt in range(1, retries + 1):\n        try:\n            req = urllib.request.Request(\n                url,\n                headers={"User-Agent": "ReliableVAEP/Canonical-Validation"},\n            )\n            with urllib.request.urlopen(req, timeout=120) as response:\n                obj = json.loads(response.read().decode("utf-8"))\n            with open(path, "w", encoding="utf-8") as f:\n                json.dump(obj, f)\n            return obj\n        except Exception as exc:\n            if attempt == retries:\n                raise RuntimeError(f"Download failed: {url}") from exc\n            time.sleep(min(2 ** attempt, 15))\n\n\ndef make_model(seed):\n    return xgb.XGBClassifier(\n        n_estimators=180,\n        max_depth=3,\n        learning_rate=0.05,\n        min_child_weight=5,\n        subsample=0.85,\n        colsample_bytree=0.85,\n        reg_lambda=2.0,\n        objective="binary:logistic",\n        eval_metric="logloss",\n        tree_method="hist",\n        random_state=seed,\n        n_jobs=-1,\n    )\n\n\ndef fit_platt(prob, y):\n    prob = np.asarray(prob, dtype=float)\n    y = np.asarray(y, dtype=int)\n\n    if len(np.unique(y)) < 2:\n        return None\n\n    z = logit(\n        np.clip(prob, 1e-6, 1 - 1e-6)\n    ).reshape(-1, 1)\n\n    model = LogisticRegression(\n        solver="lbfgs",\n        max_iter=1000,\n    )\n    model.fit(z, y)\n    return model\n\n\ndef apply_platt(model, prob):\n    prob = np.asarray(prob, dtype=float)\n\n    if model is None:\n        return prob.copy()\n\n    z = logit(\n        np.clip(prob, 1e-6, 1 - 1e-6)\n    ).reshape(-1, 1)\n\n    return model.predict_proba(z)[:, 1]\n\n\ndef safe_auc(y, p):\n    return (\n        float(roc_auc_score(y, p))\n        if len(np.unique(y)) > 1\n        else np.nan\n    )\n\n\ndef baseline_brier(y):\n    y = np.asarray(y, dtype=float)\n    return float(\n        np.mean((y - y.mean()) ** 2)\n    )\n\n\ndef vaep90(q):\n    minutes = float(q["minutes_played"].sum())\n    if minutes <= 0:\n        return np.nan\n    return float(\n        90.0 * q["vaep"].sum() / minutes\n    )\n\n\ndef threshold_crossing(summary, threshold=0.80, sustained_points=3):\n    q = summary.sort_values("n").reset_index(drop=True)\n\n    passed = (\n        q["median_rho"].to_numpy()\n        >= threshold\n    )\n\n    first = None\n    if passed.any():\n        first = int(\n            q.loc[\n                np.where(passed)[0][0],\n                "n",\n            ]\n        )\n\n    sustained = None\n    for i in range(\n        len(q) - sustained_points + 1\n    ):\n        if passed[i:i + sustained_points].all():\n            sustained = int(\n                q.loc[i, "n"]\n            )\n            break\n\n    return {\n        "threshold": float(threshold),\n        "first_crossing_n": first,\n        "sustained_crossing_n": sustained,\n        "highest_median_rho": float(\n            q["median_rho"].max()\n        ),\n        "max_tested_n": int(\n            q["n"].max()\n        ),\n    }\n\n\nSBL = StatsBombLoader(\n    getter="remote",\n    creds=None,\n)\n\n\ndef build_dataset(dataset_name, season_config):\n    dataset_out = OUT_ROOT / dataset_name\n    dataset_cache = CACHE_ROOT / dataset_name\n    dataset_out.mkdir(parents=True, exist_ok=True)\n    dataset_cache.mkdir(parents=True, exist_ok=True)\n\n    complete_marker = dataset_out / "BUILD_COMPLETE.json"\n    pm_pickle = dataset_out / "player_match_vaep.pkl"\n\n    # Safe checkpoint: only use cached final build when marker and files exist.\n    if complete_marker.exists() and pm_pickle.exists():\n        try:\n            meta = json.load(open(complete_marker, encoding="utf-8"))\n            pm = pd.read_pickle(pm_pickle)\n            all_cohort = pd.read_csv(dataset_out / "strict_cohort_all_players.csv")\n            outfield_cohort = pd.read_csv(dataset_out / "strict_cohort_outfield_only.csv")\n            cv_metrics = pd.read_csv(dataset_out / "crossfit_metrics.csv")\n\n            print(f"\\n{dataset_name}: loading completed cached dataset.")\n            print(\n                f"  all players={len(meta[\'all_ids\'])}, "\n                f"outfield={len(meta[\'outfield_ids\'])}, "\n                f"goalkeepers={len(meta[\'goalkeeper_ids\'])}"\n            )\n\n            return {\n                "dataset_name": dataset_name,\n                "player_match": pm,\n                "all_ids": meta["all_ids"],\n                "outfield_ids": meta["outfield_ids"],\n                "goalkeeper_ids": meta["goalkeeper_ids"],\n                "all_cohort": all_cohort,\n                "outfield_cohort": outfield_cohort,\n                "cv_metrics": cv_metrics,\n                "game_ids": meta["game_ids"],\n                "median_goal_vaep": meta["median_goal_vaep"],\n            }\n        except Exception as exc:\n            print("Completed-cache load failed; rebuilding:", exc)\n\n    print("\\n" + "=" * 88)\n    print("BUILDING:", dataset_name)\n    print("=" * 88)\n\n    # ------------------------------------------------------------\n    # 1. Identify Barcelona matches.\n    # ------------------------------------------------------------\n    game_rows = []\n\n    for season_name, cfg in season_config.items():\n        raw_path = (\n            dataset_cache\n            / "match_lists"\n            / f\'{cfg["competition_id"]}_{cfg["season_id"]}.json\'\n        )\n\n        raw_matches = get_json(\n            f\'{BASE}/matches/{cfg["competition_id"]}/{cfg["season_id"]}.json\',\n            raw_path,\n        )\n\n        season_count = 0\n\n        for m in raw_matches:\n            home = str(\n                m["home_team"]["home_team_name"]\n            )\n            away = str(\n                m["away_team"]["away_team_name"]\n            )\n\n            if (\n                "barcelona" not in home.lower()\n                and "barcelona" not in away.lower()\n            ):\n                continue\n\n            home_id = int(\n                m["home_team"]["home_team_id"]\n            )\n            away_id = int(\n                m["away_team"]["away_team_id"]\n            )\n            barca_id = (\n                home_id\n                if "barcelona" in home.lower()\n                else away_id\n            )\n\n            game_rows.append({\n                "dataset": dataset_name,\n                "season": season_name,\n                "game_id": int(m["match_id"]),\n                "game_date": pd.to_datetime(\n                    m["match_date"]\n                ),\n                "home_team_id": home_id,\n                "away_team_id": away_id,\n                "barca_team_id": barca_id,\n                "home_team_name": home,\n                "away_team_name": away,\n            })\n            season_count += 1\n\n        if season_count < 20:\n            raise RuntimeError(\n                f"{dataset_name} {season_name}: only {season_count} Barcelona matches found. "\n                "This is unexpectedly low and may indicate a wrong season ID or changed source."\n            )\n\n    games = (\n        pd.DataFrame(game_rows)\n        .sort_values(\n            ["game_date", "game_id"]\n        )\n        .reset_index(drop=True)\n    )\n\n    counts = (\n        games.groupby("season")["game_id"]\n        .nunique()\n    )\n\n    print("\\nBarcelona match availability:")\n    print(counts.to_string())\n    print("Total matches:", games["game_id"].nunique())\n\n    if len(counts) != 5:\n        raise RuntimeError(\n            f"{dataset_name}: expected five seasons, got {len(counts)}."\n        )\n\n    games.to_csv(\n        dataset_out / "game_list.csv",\n        index=False,\n    )\n\n    # ------------------------------------------------------------\n    # 2. SPADL conversion and official player minutes.\n    # ------------------------------------------------------------\n    store = {}\n\n    for g in tqdm(\n        games.itertuples(index=False),\n        total=len(games),\n        desc=f"{dataset_name}: SPADL",\n        unit="match",\n    ):\n        gid = int(g.game_id)\n\n        ap = (\n            dataset_cache / "spadl" / f"{gid}.pkl"\n        )\n        pp = (\n            dataset_cache / "players" / f"{gid}.pkl"\n        )\n\n        ap.parent.mkdir(\n            parents=True,\n            exist_ok=True,\n        )\n        pp.parent.mkdir(\n            parents=True,\n            exist_ok=True,\n        )\n\n        if ap.exists():\n            actions = pd.read_pickle(ap)\n        else:\n            events = SBL.events(\n                game_id=gid\n            )\n            actions = (\n                spadl.statsbomb\n                .convert_to_actions(\n                    events=events,\n                    home_team_id=int(\n                        g.home_team_id\n                    ),\n                )\n                .reset_index(drop=True)\n            )\n            actions.to_pickle(ap)\n\n        if pp.exists():\n            players = pd.read_pickle(pp)\n        else:\n            players = SBL.players(\n                game_id=gid\n            )\n            players.to_pickle(pp)\n\n        if len(actions) == 0:\n            raise RuntimeError(\n                f"No SPADL actions for game {gid}"\n            )\n\n        store[gid] = {\n            "game": pd.Series({\n                "game_id": gid,\n                "home_team_id": int(\n                    g.home_team_id\n                ),\n                "away_team_id": int(\n                    g.away_team_id\n                ),\n                "barca_team_id": int(\n                    g.barca_team_id\n                ),\n                "season": g.season,\n                "game_date": g.game_date,\n            }),\n            "actions": actions,\n            "players": players,\n        }\n\n    # ------------------------------------------------------------\n    # 3. Canonical VAEP features and labels.\n    # ------------------------------------------------------------\n    template = VAEP()\n\n    X_parts = []\n    Y_parts = []\n    row_parts = []\n\n    for gid, item in tqdm(\n        store.items(),\n        desc=f"{dataset_name}: features",\n        unit="match",\n    ):\n        Xg = (\n            template.compute_features(\n                item["game"],\n                item["actions"],\n            )\n            .reset_index(drop=True)\n        )\n\n        Yg = (\n            template.compute_labels(\n                item["game"],\n                item["actions"],\n            )\n            .reset_index(drop=True)\n        )\n\n        if (\n            len(Xg) != len(item["actions"])\n            or len(Yg) != len(item["actions"])\n        ):\n            raise RuntimeError(\n                f"Feature/label/action alignment failure in game {gid}"\n            )\n\n        X_parts.append(Xg)\n        Y_parts.append(Yg)\n\n        row_parts.append(\n            pd.DataFrame({\n                "game_id": gid,\n                "action_row": np.arange(\n                    len(Xg)\n                ),\n            })\n        )\n\n    X = pd.concat(\n        X_parts,\n        ignore_index=True,\n    )\n    Y = pd.concat(\n        Y_parts,\n        ignore_index=True,\n    )\n    ROW = pd.concat(\n        row_parts,\n        ignore_index=True,\n    )\n\n    for col in X.columns:\n        if pd.api.types.is_bool_dtype(\n            X[col]\n        ):\n            X[col] = (\n                X[col]\n                .astype(np.uint8)\n            )\n        elif isinstance(\n            X[col].dtype,\n            pd.CategoricalDtype,\n        ):\n            X[col] = (\n                X[col]\n                .cat.codes\n                .astype(np.int32)\n            )\n        elif pd.api.types.is_object_dtype(\n            X[col]\n        ):\n            X[col] = (\n                pd.factorize(\n                    X[col].astype(str)\n                )[0]\n                .astype(np.int32)\n            )\n\n    bad_features = [\n        c for c in X.columns\n        if not pd.api.types.is_numeric_dtype(\n            X[c]\n        )\n    ]\n\n    if bad_features:\n        raise TypeError(\n            f"Non-numeric VAEP feature columns remain: {bad_features}"\n        )\n\n    print(\n        f"Feature rows={len(X):,}, "\n        f"features={X.shape[1]}, "\n        f"score rate={Y[\'scores\'].mean():.4f}, "\n        f"concede rate={Y[\'concedes\'].mean():.4f}"\n    )\n\n    # ------------------------------------------------------------\n    # 4. Match-level cross-fitting + separate calibration matches.\n    # ------------------------------------------------------------\n    groups = (\n        ROW["game_id"]\n        .to_numpy()\n    )\n\n    unique_matches = (\n        np.unique(groups)\n    )\n\n    n_folds = min(\n        OUTER_FOLDS,\n        len(unique_matches),\n    )\n\n    if n_folds < 3:\n        raise RuntimeError(\n            "Too few matches for grouped cross-fitting."\n        )\n\n    outer = GroupKFold(\n        n_splits=n_folds\n    )\n\n    oof = {\n        "scores_raw": np.full(\n            len(X),\n            np.nan,\n        ),\n        "scores_cal": np.full(\n            len(X),\n            np.nan,\n        ),\n        "concedes_raw": np.full(\n            len(X),\n            np.nan,\n        ),\n        "concedes_cal": np.full(\n            len(X),\n            np.nan,\n        ),\n    }\n\n    metric_rows = []\n\n    for fold, (\n        train_idx,\n        test_idx,\n    ) in enumerate(\n        outer.split(\n            X,\n            groups=groups,\n        ),\n        start=1,\n    ):\n        inner = GroupShuffleSplit(\n            n_splits=1,\n            test_size=CALIBRATION_MATCH_FRACTION,\n            random_state=SEED + fold,\n        )\n\n        fit_rel, cal_rel = next(\n            inner.split(\n                X.iloc[train_idx],\n                groups=groups[\n                    train_idx\n                ],\n            )\n        )\n\n        fit_idx = (\n            train_idx[fit_rel]\n        )\n        cal_idx = (\n            train_idx[cal_rel]\n        )\n\n        row = {\n            "dataset": dataset_name,\n            "fold": fold,\n            "fit_matches": int(\n                len(\n                    np.unique(\n                        groups[fit_idx]\n                    )\n                )\n            ),\n            "calibration_matches": int(\n                len(\n                    np.unique(\n                        groups[cal_idx]\n                    )\n                )\n            ),\n            "test_matches": int(\n                len(\n                    np.unique(\n                        groups[test_idx]\n                    )\n                )\n            ),\n        }\n\n        for target in [\n            "scores",\n            "concedes",\n        ]:\n            model = make_model(\n                SEED\n                + fold * 10\n                + (\n                    1\n                    if target == "scores"\n                    else 2\n                )\n            )\n\n            model.fit(\n                X.iloc[fit_idx],\n                Y.iloc[fit_idx][target]\n                .astype(int),\n            )\n\n            p_cal_raw = (\n                model.predict_proba(\n                    X.iloc[cal_idx]\n                )[:, 1]\n            )\n\n            calibrator = fit_platt(\n                p_cal_raw,\n                Y.iloc[cal_idx][target]\n                .astype(int),\n            )\n\n            p_test_raw = (\n                model.predict_proba(\n                    X.iloc[test_idx]\n                )[:, 1]\n            )\n\n            p_test_cal = (\n                apply_platt(\n                    calibrator,\n                    p_test_raw,\n                )\n            )\n\n            oof[\n                f"{target}_raw"\n            ][test_idx] = (\n                p_test_raw\n            )\n\n            oof[\n                f"{target}_cal"\n            ][test_idx] = (\n                p_test_cal\n            )\n\n            y_test = (\n                Y.iloc[test_idx][target]\n                .astype(int)\n                .to_numpy()\n            )\n\n            base = baseline_brier(\n                y_test\n            )\n\n            row[\n                f"{target}_auc"\n            ] = safe_auc(\n                y_test,\n                p_test_cal,\n            )\n\n            row[\n                f"{target}_ap"\n            ] = float(\n                average_precision_score(\n                    y_test,\n                    p_test_cal,\n                )\n            )\n\n            row[\n                f"{target}_brier_baseline"\n            ] = base\n\n            row[\n                f"{target}_brier_raw"\n            ] = float(\n                brier_score_loss(\n                    y_test,\n                    p_test_raw,\n                )\n            )\n\n            row[\n                f"{target}_brier_cal"\n            ] = float(\n                brier_score_loss(\n                    y_test,\n                    p_test_cal,\n                )\n            )\n\n            row[\n                f"{target}_brier_skill_cal"\n            ] = (\n                1.0\n                - row[\n                    f"{target}_brier_cal"\n                ]\n                / base\n                if base > 0\n                else np.nan\n            )\n\n        metric_rows.append(row)\n\n    for key, values in oof.items():\n        if np.isnan(values).any():\n            raise RuntimeError(\n                f"Incomplete OOF predictions: {key}"\n            )\n\n    cv_metrics = pd.DataFrame(\n        metric_rows\n    )\n\n    print("\\nCross-fit model means:")\n    metric_cols = [\n        c for c in cv_metrics.columns\n        if c not in {\n            "dataset",\n            "fold",\n        }\n    ]\n    print(\n        cv_metrics[\n            metric_cols\n        ]\n        .mean(\n            numeric_only=True\n        )\n        .round(4)\n        .to_string()\n    )\n\n    # ------------------------------------------------------------\n    # 5. Canonical action VAEP.\n    # ------------------------------------------------------------\n    scored_parts = []\n    cursor = 0\n\n    for gid, item in tqdm(\n        store.items(),\n        desc=f"{dataset_name}: VAEP values",\n        unit="match",\n    ):\n        actions = (\n            item["actions"]\n            .reset_index(drop=True)\n        )\n\n        n_actions = len(\n            actions\n        )\n\n        ps = pd.Series(\n            oof["scores_cal"][\n                cursor:cursor + n_actions\n            ]\n        )\n\n        pc = pd.Series(\n            oof["concedes_cal"][\n                cursor:cursor + n_actions\n            ]\n        )\n\n        named = (\n            spadl.add_names(\n                actions.copy()\n            )\n            .reset_index(drop=True)\n        )\n\n        values = (\n            vaep_formula.value(\n                named,\n                ps,\n                pc,\n            )\n            .reset_index(drop=True)\n        )\n\n        z = named.copy()\n\n        z["game_id"] = gid\n        z["season"] = (\n            item["game"]["season"]\n        )\n        z["game_date"] = (\n            item["game"]["game_date"]\n        )\n        z["barca_team_id"] = int(\n            item["game"]["barca_team_id"]\n        )\n        z["p_score"] = (\n            ps.to_numpy()\n        )\n        z["p_concede"] = (\n            pc.to_numpy()\n        )\n\n        for col in [\n            "offensive_value",\n            "defensive_value",\n            "vaep_value",\n        ]:\n            z[col] = (\n                values[col]\n                .to_numpy()\n            )\n\n        scored_parts.append(\n            z\n        )\n\n        cursor += (\n            n_actions\n        )\n\n    if cursor != len(X):\n        raise RuntimeError(\n            "OOF probability/action alignment failed."\n        )\n\n    scored = pd.concat(\n        scored_parts,\n        ignore_index=True,\n    )\n\n    # ------------------------------------------------------------\n    # 6. Goal-action sanity check.\n    # ------------------------------------------------------------\n    goal_mask = (\n        scored["type_name"].isin([\n            "shot",\n            "shot_freekick",\n            "shot_penalty",\n        ])\n        & scored[\n            "result_name"\n        ].eq("success")\n    )\n\n    if int(\n        goal_mask.sum()\n    ) == 0:\n        raise RuntimeError(\n            "No successful goal actions detected."\n        )\n\n    goal_values = (\n        scored.loc[\n            goal_mask,\n            "vaep_value",\n        ]\n    )\n\n    median_goal_vaep = float(\n        goal_values.median()\n    )\n\n    positive_goal_fraction = float(\n        (goal_values > 0).mean()\n    )\n\n    print(\n        "\\nGoal sanity check:",\n        f"n={int(goal_mask.sum())}, "\n        f"median VAEP={median_goal_vaep:.4f}, "\n        f"positive fraction={positive_goal_fraction:.3f}"\n    )\n\n    if median_goal_vaep <= 0:\n        raise RuntimeError(\n            "Canonical VAEP sanity check failed: median successful-goal VAEP is not positive."\n        )\n\n    # ------------------------------------------------------------\n    # 7. Player-match table using official player minutes.\n    # ------------------------------------------------------------\n    pm_parts = []\n\n    for gid, item in tqdm(\n        store.items(),\n        desc=f"{dataset_name}: player-match",\n        unit="match",\n    ):\n        barca_id = int(\n            item["game"]["barca_team_id"]\n        )\n\n        players = (\n            item["players"]\n            .copy()\n        )\n\n        players = players[\n            players["team_id"]\n            .astype(int)\n            == barca_id\n        ].copy()\n\n        players = players[\n            players[\n                "minutes_played"\n            ].fillna(0)\n            > 0\n        ].copy()\n\n        ga = scored[\n            (\n                scored["game_id"]\n                == gid\n            )\n            & (\n                scored["team_id"]\n                .astype(int)\n                == barca_id\n            )\n        ].copy()\n\n        agg = (\n            ga.dropna(\n                subset=[\n                    "player_id"\n                ]\n            )\n            .groupby(\n                "player_id",\n                as_index=False,\n            )\n            .agg(\n                vaep=(\n                    "vaep_value",\n                    "sum",\n                ),\n                offensive_value=(\n                    "offensive_value",\n                    "sum",\n                ),\n                defensive_value=(\n                    "defensive_value",\n                    "sum",\n                ),\n                actions=(\n                    "action_id",\n                    "size",\n                ),\n            )\n        )\n\n        goals = (\n            ga[\n                ga[\n                    "type_name"\n                ].isin([\n                    "shot",\n                    "shot_freekick",\n                    "shot_penalty",\n                ])\n                & ga[\n                    "result_name"\n                ].eq("success")\n            ]\n            .dropna(\n                subset=[\n                    "player_id"\n                ]\n            )\n            .groupby(\n                "player_id"\n            )\n            .size()\n            .rename("goals")\n            .reset_index()\n        )\n\n        agg = agg.merge(\n            goals,\n            on="player_id",\n            how="left",\n        )\n\n        agg["goals"] = (\n            agg["goals"]\n            .fillna(0)\n            .astype(int)\n        )\n\n        q = players.merge(\n            agg,\n            on="player_id",\n            how="left",\n        )\n\n        for col in [\n            "vaep",\n            "offensive_value",\n            "defensive_value",\n        ]:\n            q[col] = (\n                q[col]\n                .fillna(0.0)\n            )\n\n        for col in [\n            "actions",\n            "goals",\n        ]:\n            q[col] = (\n                q[col]\n                .fillna(0)\n                .astype(int)\n            )\n\n        q["dataset"] = (\n            dataset_name\n        )\n        q["game_id"] = gid\n        q["season"] = (\n            item["game"]["season"]\n        )\n        q["game_date"] = (\n            item["game"]["game_date"]\n        )\n        q["vaep90_match"] = (\n            90.0\n            * q["vaep"]\n            / q["minutes_played"]\n        )\n\n        if (\n            "starting_position_name"\n            in q.columns\n        ):\n            q[\n                "is_goalkeeper_position"\n            ] = (\n                q[\n                    "starting_position_name"\n                ]\n                .astype(str)\n                .str.contains(\n                    "Goalkeeper",\n                    case=False,\n                    na=False,\n                )\n            )\n        else:\n            q[\n                "is_goalkeeper_position"\n            ] = False\n\n        pm_parts.append(q)\n\n    pm = pd.concat(\n        pm_parts,\n        ignore_index=True,\n    )\n\n    # Secondary goalkeeper detection from canonical goalkeeper actions.\n    keeper_types = {\n        "keeper_save",\n        "keeper_claim",\n        "keeper_punch",\n        "keeper_pick_up",\n    }\n\n    keeper_action_ids = set(\n        scored.loc[\n            scored[\n                "type_name"\n            ].isin(\n                keeper_types\n            ),\n            "player_id",\n        ]\n        .dropna()\n        .tolist()\n    )\n\n    pm[\n        "is_goalkeeper"\n    ] = (\n        pm[\n            "is_goalkeeper_position"\n        ].astype(bool)\n        | pm[\n            "player_id"\n        ].isin(\n            keeper_action_ids\n        )\n    )\n\n    # ------------------------------------------------------------\n    # 8. Strict five-season cohorts.\n    # ------------------------------------------------------------\n    season_sets = []\n\n    for season in season_config:\n        ids = set(\n            pm.loc[\n                pm["season"] == season,\n                "player_id",\n            ].unique()\n        )\n        season_sets.append(ids)\n\n    all_ids = sorted(\n        set.intersection(\n            *season_sets\n        )\n    )\n\n    if len(all_ids) < 4:\n        raise RuntimeError(\n            f"{dataset_name}: strict five-season cohort has only {len(all_ids)} players."\n        )\n\n    goalkeeper_ids = sorted(\n        set(\n            pm.loc[\n                (\n                    pm["player_id"]\n                    .isin(\n                        all_ids\n                    )\n                )\n                & (\n                    pm[\n                        "is_goalkeeper"\n                    ]\n                ),\n                "player_id",\n            ].unique()\n        )\n    )\n\n    outfield_ids = [\n        pid\n        for pid in all_ids\n        if pid not in goalkeeper_ids\n    ]\n\n    if len(outfield_ids) < 4:\n        raise RuntimeError(\n            f"{dataset_name}: outfield strict cohort has only {len(outfield_ids)} players."\n        )\n\n    if len(goalkeeper_ids) != 1:\n        print(\n            f"WARNING: strict cohort contains {len(goalkeeper_ids)} detected goalkeeper(s), "\n            "not exactly one. All detected goalkeepers will be excluded from outfield analysis."\n        )\n\n    def build_cohort(ids, analysis_name):\n        c = (\n            pm[\n                pm["player_id"]\n                .isin(ids)\n            ]\n            .groupby(\n                [\n                    "player_id",\n                    "player_name",\n                ],\n                as_index=False,\n            )\n            .agg(\n                seasons=(\n                    "season",\n                    "nunique",\n                ),\n                total_matches=(\n                    "game_id",\n                    "nunique",\n                ),\n                total_minutes=(\n                    "minutes_played",\n                    "sum",\n                ),\n                total_actions=(\n                    "actions",\n                    "sum",\n                ),\n                goals=(\n                    "goals",\n                    "sum",\n                ),\n                offensive_value=(\n                    "offensive_value",\n                    "sum",\n                ),\n                defensive_value=(\n                    "defensive_value",\n                    "sum",\n                ),\n                total_vaep=(\n                    "vaep",\n                    "sum",\n                ),\n            )\n        )\n\n        c[\n            "vaep90_all"\n        ] = (\n            90.0\n            * c["total_vaep"]\n            / c["total_minutes"]\n        )\n\n        c = (\n            c.sort_values(\n                "vaep90_all",\n                ascending=False,\n            )\n            .reset_index(drop=True)\n        )\n\n        c[\n            "full_data_rank"\n        ] = np.arange(\n            1,\n            len(c) + 1,\n        )\n\n        c[\n            "analysis"\n        ] = analysis_name\n\n        return c\n\n    all_cohort = build_cohort(\n        all_ids,\n        "all_players",\n    )\n\n    outfield_cohort = build_cohort(\n        outfield_ids,\n        "outfield_only",\n    )\n\n    # Sanity: every retained player must be present in all five seasons.\n    if not (\n        all_cohort[\n            "seasons"\n        ].eq(5).all()\n    ):\n        raise RuntimeError(\n            "Strict all-player cohort does not have five-season coverage."\n        )\n\n    if not (\n        outfield_cohort[\n            "seasons"\n        ].eq(5).all()\n    ):\n        raise RuntimeError(\n            "Strict outfield cohort does not have five-season coverage."\n        )\n\n    print("\\nStrict all-player cohort:")\n    print(\n        all_cohort[\n            [\n                "player_name",\n                "total_matches",\n                "total_minutes",\n                "goals",\n                "vaep90_all",\n                "full_data_rank",\n            ]\n        ]\n        .round(4)\n        .to_string(\n            index=False\n        )\n    )\n\n    print(\n        "\\nDetected strict-cohort goalkeeper IDs:",\n        goalkeeper_ids\n    )\n\n    print(\n        "All-player count:",\n        len(all_ids),\n        "| Outfield-only count:",\n        len(outfield_ids)\n    )\n\n    # ------------------------------------------------------------\n    # 9. Save completed build.\n    # ------------------------------------------------------------\n    cv_metrics.to_csv(\n        dataset_out\n        / "crossfit_metrics.csv",\n        index=False,\n    )\n\n    pm.to_pickle(\n        pm_pickle\n    )\n\n    pm.to_csv(\n        dataset_out\n        / "player_match_vaep.csv",\n        index=False,\n    )\n\n    all_cohort.to_csv(\n        dataset_out\n        / "strict_cohort_all_players.csv",\n        index=False,\n    )\n\n    outfield_cohort.to_csv(\n        dataset_out\n        / "strict_cohort_outfield_only.csv",\n        index=False,\n    )\n\n    meta = {\n        "all_ids": [\n            int(x)\n            if isinstance(\n                x,\n                (\n                    np.integer,\n                    int,\n                )\n            )\n            else float(x)\n            for x in all_ids\n        ],\n        "outfield_ids": [\n            int(x)\n            if isinstance(\n                x,\n                (\n                    np.integer,\n                    int,\n                )\n            )\n            else float(x)\n            for x in outfield_ids\n        ],\n        "goalkeeper_ids": [\n            int(x)\n            if isinstance(\n                x,\n                (\n                    np.integer,\n                    int,\n                )\n            )\n            else float(x)\n            for x in goalkeeper_ids\n        ],\n        "game_ids": [\n            int(x)\n            for x in games[\n                "game_id"\n            ].tolist()\n        ],\n        "median_goal_vaep": median_goal_vaep,\n    }\n\n    with open(\n        complete_marker,\n        "w",\n        encoding="utf-8",\n    ) as f:\n        json.dump(\n            meta,\n            f,\n            indent=2,\n        )\n\n    return {\n        "dataset_name": dataset_name,\n        "player_match": pm,\n        "all_ids": all_ids,\n        "outfield_ids": outfield_ids,\n        "goalkeeper_ids": goalkeeper_ids,\n        "all_cohort": all_cohort,\n        "outfield_cohort": outfield_cohort,\n        "cv_metrics": cv_metrics,\n        "game_ids": games[\n            "game_id"\n        ].tolist(),\n        "median_goal_vaep": median_goal_vaep,\n    }\n\n\n\n\n# ================================================================\n# PCS / RAR: FULL-DATA STATIC RANKING + INDEPENDENT REPRODUCIBILITY\n# ================================================================\n\nPAIRWISE_TAU = 0.90\nRIDGE = 1e-4\n\n# Publication-quality defaults.\n# QUICK_TEST=True is only for debugging the pipeline.\nQUICK_TEST = False\n\nif QUICK_TEST:\n    FULL_BOOTSTRAP_B = 300\n    VALIDATION_REPS = 20\n    INNER_BOOTSTRAP_B = 100\nelse:\n    FULL_BOOTSTRAP_B = 3500\n    VALIDATION_REPS = 200\n    INNER_BOOTSTRAP_B = 500\n\n# We are testing whether RAR is more reproducible than raw VAEP.\n# A 5-match grid is sufficient for the adjustment comparison and\n# avoids needlessly fitting tens of thousands of near-duplicate BT models.\nN_START = 5\nN_STEP = 5\n\n\ndef pair_prob_gt(x, y):\n    x = np.asarray(x, dtype=float)\n    y = np.asarray(y, dtype=float)\n    valid = np.isfinite(x) & np.isfinite(y)\n    if valid.sum() == 0:\n        return np.nan\n    d = x[valid] - y[valid]\n    return float(\n        np.mean(\n            np.where(\n                d > 0,\n                1.0,\n                np.where(d == 0, 0.5, 0.0),\n            )\n        )\n    )\n\n\ndef pairwise_probability_matrix(value_draws):\n    value_draws = np.asarray(value_draws, dtype=float)\n    K = value_draws.shape[1]\n    P = np.full((K, K), 0.5, dtype=float)\n\n    for i in range(K):\n        for j in range(i + 1, K):\n            p = pair_prob_gt(value_draws[:, i], value_draws[:, j])\n            P[i, j] = p\n            P[j, i] = 1.0 - p if np.isfinite(p) else np.nan\n\n    return P\n\n\ndef fit_pairwise_consensus(P, ridge=RIDGE):\n    """\n    Bradley-Terry-style soft-label fit:\n        P(i > j) ~= sigmoid(S_i - S_j)\n\n    PCS is the fitted latent score S.\n    RAR is the deterministic rank induced by PCS.\n    """\n    P = np.asarray(P, dtype=float)\n    K = P.shape[0]\n\n    if K == 1:\n        return np.array([0.0]), {\n            "success": True,\n            "pairwise_mae": 0.0,\n            "pairwise_rmse": 0.0,\n            "objective": 0.0,\n        }\n\n    ii, jj = np.triu_indices(K, k=1)\n    target = P[ii, jj]\n    valid = np.isfinite(target)\n    ii, jj, target = ii[valid], jj[valid], target[valid]\n\n    if len(target) == 0:\n        return np.zeros(K), {\n            "success": False,\n            "pairwise_mae": np.nan,\n            "pairwise_rmse": np.nan,\n            "objective": np.nan,\n        }\n\n    def loss_grad(s):\n        z = s[ii] - s[jj]\n        # Cross entropy for soft pairwise probabilities.\n        losses = np.logaddexp(0.0, z) - target * z\n        loss = float(np.mean(losses) + ridge * np.mean(s ** 2))\n\n        err = (expit(z) - target) / len(target)\n        grad = np.zeros_like(s)\n        np.add.at(grad, ii, err)\n        np.add.at(grad, jj, -err)\n        grad += (2.0 * ridge / K) * s\n        return loss, grad\n\n    result = minimize(\n        fun=lambda s: loss_grad(s)[0],\n        x0=np.zeros(K, dtype=float),\n        jac=lambda s: loss_grad(s)[1],\n        method="L-BFGS-B",\n        options={"maxiter": 1000, "ftol": 1e-12},\n    )\n\n    score = result.x - np.mean(result.x)\n    pred = expit(score[ii] - score[jj])\n\n    return score, {\n        "success": bool(result.success),\n        "pairwise_mae": float(np.mean(np.abs(pred - target))),\n        "pairwise_rmse": float(np.sqrt(np.mean((pred - target) ** 2))),\n        "objective": float(result.fun),\n    }\n\n\ndef bootstrap_player_match_values(pm, player_ids, B, seed):\n    """\n    Match-level player-preserving bootstrap.\n    Each player is resampled from their own observed matches.\n    """\n    rng = np.random.default_rng(seed)\n    player_ids = list(player_ids)\n    V = np.full((B, len(player_ids)), np.nan, dtype=float)\n\n    grouped = {\n        pid: q.reset_index(drop=True)\n        for pid, q in pm[pm["player_id"].isin(player_ids)].groupby("player_id")\n    }\n\n    for j, pid in enumerate(player_ids):\n        q = grouped[pid]\n        va = q["vaep"].to_numpy(dtype=float)\n        mi = q["minutes_played"].to_numpy(dtype=float)\n        m = len(q)\n\n        draws = rng.integers(0, m, size=(B, m))\n        total_v = va[draws].sum(axis=1)\n        total_m = mi[draws].sum(axis=1)\n\n        V[:, j] = (\n            90.0\n            * total_v\n            / np.where(total_m > 0, total_m, np.nan)\n        )\n\n    return V\n\n\ndef raw_values_from_pm(pm, player_ids):\n    vals = []\n    for pid in player_ids:\n        q = pm[pm["player_id"] == pid]\n        vals.append(vaep90(q))\n    return np.asarray(vals, dtype=float)\n\n\ndef ranks_from_values(values):\n    return (\n        pd.Series(np.asarray(values, dtype=float))\n        .rank(ascending=False, method="average")\n        .to_numpy()\n    )\n\n\ndef rar_from_draws(draws):\n    P = pairwise_probability_matrix(draws)\n    score, fit = fit_pairwise_consensus(P, ridge=RIDGE)\n    rar = (\n        pd.Series(score)\n        .rank(ascending=False, method="average")\n        .to_numpy()\n    )\n    return score, rar, P, fit\n\n\ndef static_consensus_leaderboard(\n    pm,\n    cohort_table,\n    player_ids,\n    experiment_key,\n    B=FULL_BOOTSTRAP_B,\n    seed=SEED + 500,\n):\n    """\n    Full observed-data static output:\n    VAEP/90 + raw rank + PCS + RAR + supported rank band.\n    """\n    player_ids = list(player_ids)\n\n    names = (\n        cohort_table.set_index("player_id")["player_name"]\n        .reindex(player_ids)\n    )\n\n    raw_values = raw_values_from_pm(pm, player_ids)\n    raw_rank = ranks_from_values(raw_values)\n\n    draws = bootstrap_player_match_values(\n        pm,\n        player_ids,\n        B=B,\n        seed=seed,\n    )\n\n    score, rar, P, fit = rar_from_draws(draws)\n    K = len(player_ids)\n\n    if np.std(score) > 0:\n        pcs_z = (score - np.mean(score)) / np.std(score)\n    else:\n        pcs_z = np.zeros_like(score)\n\n    player_rows = []\n    pair_rows = []\n\n    for i in range(K):\n        other = np.array([j for j in range(K) if j != i], dtype=int)\n        p_i = P[i, other]\n\n        # Players strongly above player i:\n        n_strongly_above_i = int(np.sum(p_i <= (1.0 - PAIRWISE_TAU)))\n        # Players player i is strongly above:\n        n_strongly_below_i = int(np.sum(p_i >= PAIRWISE_TAU))\n\n        low = 1 + n_strongly_above_i\n        high = K - n_strongly_below_i\n\n        resolved = (\n            (p_i >= PAIRWISE_TAU)\n            | (p_i <= (1.0 - PAIRWISE_TAU))\n        )\n\n        player_rows.append({\n            "experiment": experiment_key,\n            "player_id": player_ids[i],\n            "player_name": names.loc[player_ids[i]],\n            "vaep90": float(raw_values[i]),\n            "raw_VAEP_rank": float(raw_rank[i]),\n            "PCS": float(score[i]),\n            "PCS_z": float(pcs_z[i]),\n            "RAR": float(rar[i]),\n            "RAR_minus_raw_rank": float(rar[i] - raw_rank[i]),\n            "supported_rank_low": int(low),\n            "supported_rank_high": int(high),\n            "supported_rank_width": int(high - low),\n            "pairwise_resolution_rate": float(np.mean(resolved)) if len(other) else 1.0,\n            "mean_pairwise_certainty": float(\n                np.mean(2.0 * np.abs(p_i - 0.5))\n            ) if len(other) else 1.0,\n            "bootstrap_draws": int(B),\n            "support_threshold": float(PAIRWISE_TAU),\n        })\n\n    for i in range(K):\n        for j in range(i + 1, K):\n            p = float(P[i, j])\n            if p >= PAIRWISE_TAU:\n                status = "i_above_j_supported"\n            elif p <= (1.0 - PAIRWISE_TAU):\n                status = "j_above_i_supported"\n            else:\n                status = "unresolved"\n\n            pair_rows.append({\n                "experiment": experiment_key,\n                "player_i_id": player_ids[i],\n                "player_i_name": names.loc[player_ids[i]],\n                "player_j_id": player_ids[j],\n                "player_j_name": names.loc[player_ids[j]],\n                "P_i_above_j": p,\n                "support_status": status,\n                "BT_pred_P_i_above_j": float(expit(score[i] - score[j])),\n                "raw_i_above_j": bool(raw_rank[i] < raw_rank[j]),\n                "RAR_i_above_j": bool(rar[i] < rar[j]),\n            })\n\n    leaderboard = (\n        pd.DataFrame(player_rows)\n        .sort_values(["RAR", "raw_VAEP_rank"])\n        .reset_index(drop=True)\n    )\n\n    pairs = pd.DataFrame(pair_rows)\n\n    fit_row = {\n        "experiment": experiment_key,\n        "players": K,\n        "BT_fit_success": fit["success"],\n        "BT_pairwise_mae": fit["pairwise_mae"],\n        "BT_pairwise_rmse": fit["pairwise_rmse"],\n        "changed_rank_fraction": float(\n            np.mean(raw_rank != rar)\n        ),\n        "median_absolute_rank_shift": float(\n            np.median(np.abs(raw_rank - rar))\n        ),\n        "resolved_pair_fraction": float(\n            np.mean(\n                (pairs["P_i_above_j"] >= PAIRWISE_TAU)\n                | (pairs["P_i_above_j"] <= (1.0 - PAIRWISE_TAU))\n            )\n        ) if len(pairs) else 1.0,\n    }\n\n    return leaderboard, pairs, pd.DataFrame([fit_row])\n\n\ndef bootstrap_values_from_fixed_subset(q, B, rng):\n    """\n    Bootstrap a fixed n-match sample for one player.\n    """\n    q = q.reset_index(drop=True)\n    va = q["vaep"].to_numpy(dtype=float)\n    mi = q["minutes_played"].to_numpy(dtype=float)\n    n = len(q)\n\n    idx = rng.integers(0, n, size=(B, n))\n    total_v = va[idx].sum(axis=1)\n    total_m = mi[idx].sum(axis=1)\n\n    return 90.0 * total_v / np.where(total_m > 0, total_m, np.nan)\n\n\ndef independent_rar_validation(\n    pm,\n    player_ids,\n    experiment_key,\n    reps=VALIDATION_REPS,\n    inner_B=INNER_BOOTSTRAP_B,\n    seed=SEED + 1000,\n):\n    """\n    Critical validation.\n\n    At each n:\n      - sample 2n matches without replacement for every fixed player;\n      - split into non-overlapping sample A and B;\n      - calculate raw VAEP rankings independently;\n      - bootstrap sample A and sample B independently;\n      - fit PCS/RAR independently;\n      - compare raw A vs raw B and RAR A vs RAR B using Spearman.\n    """\n    rng = np.random.default_rng(seed)\n    player_ids = list(player_ids)\n\n    grouped = {\n        pid: (\n            q.sort_values(["game_date", "game_id"])\n            .reset_index(drop=True)\n        )\n        for pid, q in pm[pm["player_id"].isin(player_ids)].groupby("player_id")\n    }\n\n    if set(grouped) != set(player_ids):\n        missing = set(player_ids) - set(grouped)\n        raise RuntimeError(\n            f"{experiment_key}: missing player-match records for {missing}"\n        )\n\n    min_matches = min(len(grouped[pid]) for pid in player_ids)\n    n_max = min_matches // 2\n\n    if n_max < N_START:\n        raise RuntimeError(\n            f"{experiment_key}: n_max={n_max}, too small."\n        )\n\n    n_values = list(range(N_START, n_max + 1, N_STEP))\n    if n_values[-1] != n_max:\n        n_values.append(n_max)\n\n    rows = []\n\n    for n in tqdm(\n        n_values,\n        desc=f"{experiment_key}: VAEP vs RAR",\n        unit="n",\n    ):\n        for rep in range(reps):\n\n            raw_left = np.empty(len(player_ids), dtype=float)\n            raw_right = np.empty(len(player_ids), dtype=float)\n\n            draws_left = np.empty(\n                (inner_B, len(player_ids)),\n                dtype=float,\n            )\n            draws_right = np.empty(\n                (inner_B, len(player_ids)),\n                dtype=float,\n            )\n\n            # Each replicate has truly non-overlapping left/right match sets.\n            for j, pid in enumerate(player_ids):\n                q = grouped[pid]\n\n                chosen = rng.choice(\n                    len(q),\n                    size=2 * n,\n                    replace=False,\n                )\n\n                q_left = q.iloc[chosen[:n]].copy()\n                q_right = q.iloc[chosen[n:]].copy()\n\n                raw_left[j] = vaep90(q_left)\n                raw_right[j] = vaep90(q_right)\n\n                # Separate inner RNG streams for A and B.\n                rng_l = np.random.default_rng(\n                    seed + 10_000_000 + n * 100_000 + rep * 100 + j * 2\n                )\n                rng_r = np.random.default_rng(\n                    seed + 20_000_000 + n * 100_000 + rep * 100 + j * 2 + 1\n                )\n\n                draws_left[:, j] = bootstrap_values_from_fixed_subset(\n                    q_left, inner_B, rng_l\n                )\n                draws_right[:, j] = bootstrap_values_from_fixed_subset(\n                    q_right, inner_B, rng_r\n                )\n\n            raw_rank_left = ranks_from_values(raw_left)\n            raw_rank_right = ranks_from_values(raw_right)\n\n            _, rar_left, _, fit_l = rar_from_draws(draws_left)\n            _, rar_right, _, fit_r = rar_from_draws(draws_right)\n\n            raw_rho = float(\n                spearmanr(raw_rank_left, raw_rank_right).statistic\n            )\n            rar_rho = float(\n                spearmanr(rar_left, rar_right).statistic\n            )\n\n            rows.append({\n                "experiment": experiment_key,\n                "n_matches_per_sample": n,\n                "rep": rep,\n                "players": len(player_ids),\n                "raw_VAEP_spearman": raw_rho,\n                "RAR_spearman": rar_rho,\n                "RAR_minus_raw": rar_rho - raw_rho,\n                "RAR_better": int(rar_rho > raw_rho),\n                "RAR_equal": int(np.isclose(rar_rho, raw_rho)),\n                "RAR_worse": int(rar_rho < raw_rho),\n                "BT_left_success": bool(fit_l["success"]),\n                "BT_right_success": bool(fit_r["success"]),\n            })\n\n    raw = pd.DataFrame(rows)\n\n    summary_rows = []\n\n    for n, q in raw.groupby("n_matches_per_sample"):\n        d = q["RAR_minus_raw"].to_numpy(dtype=float)\n        nonzero = d[~np.isclose(d, 0.0)]\n\n        if len(nonzero) == 0:\n            p_value = 1.0\n        else:\n            try:\n                p_value = float(\n                    wilcoxon(\n                        nonzero,\n                        alternative="two-sided",\n                        zero_method="wilcox",\n                    ).pvalue\n                )\n            except Exception:\n                p_value = np.nan\n\n        summary_rows.append({\n            "experiment": experiment_key,\n            "n_matches_per_sample": int(n),\n            "players": int(q["players"].iloc[0]),\n            "reps": int(len(q)),\n            "median_raw_VAEP_spearman": float(q["raw_VAEP_spearman"].median()),\n            "median_RAR_spearman": float(q["RAR_spearman"].median()),\n            "median_RAR_minus_raw": float(np.median(d)),\n            "mean_RAR_minus_raw": float(np.mean(d)),\n            "q025_RAR_minus_raw": float(np.quantile(d, 0.025)),\n            "q975_RAR_minus_raw": float(np.quantile(d, 0.975)),\n            "fraction_RAR_better": float(np.mean(d > 0)),\n            "fraction_equal": float(np.mean(np.isclose(d, 0.0))),\n            "fraction_RAR_worse": float(np.mean(d < 0)),\n            "paired_wilcoxon_p": p_value,\n        })\n\n    summary = pd.DataFrame(summary_rows)\n\n    overall = pd.DataFrame([{\n        "experiment": experiment_key,\n        "players": len(player_ids),\n        "minimum_matches_in_cohort": int(min_matches),\n        "maximum_testable_n": int(n_max),\n        "tested_n_points": int(len(summary)),\n        "validation_reps_per_n": int(reps),\n        "inner_bootstrap_draws": int(inner_B),\n        "n_points_RAR_higher_median": int(\n            np.sum(\n                summary["median_RAR_spearman"]\n                > summary["median_raw_VAEP_spearman"]\n            )\n        ),\n        "n_points_equal_median": int(\n            np.sum(\n                np.isclose(\n                    summary["median_RAR_spearman"],\n                    summary["median_raw_VAEP_spearman"],\n                )\n            )\n        ),\n        "n_points_RAR_lower_median": int(\n            np.sum(\n                summary["median_RAR_spearman"]\n                < summary["median_raw_VAEP_spearman"]\n            )\n        ),\n        "pooled_median_RAR_minus_raw": float(\n            raw["RAR_minus_raw"].median()\n        ),\n        "pooled_mean_RAR_minus_raw": float(\n            raw["RAR_minus_raw"].mean()\n        ),\n        "pooled_fraction_RAR_better": float(\n            np.mean(raw["RAR_minus_raw"] > 0)\n        ),\n        "pooled_fraction_equal": float(\n            np.mean(np.isclose(raw["RAR_minus_raw"], 0.0))\n        ),\n        "pooled_fraction_RAR_worse": float(\n            np.mean(raw["RAR_minus_raw"] < 0)\n        ),\n    }])\n\n    return summary, raw, overall\n\n\n# ================================================================\n# BUILD TWO NON-OVERLAPPING CANONICAL VAEP DATASETS\n# ================================================================\n\nprimary = build_dataset(\n    "primary_2015_16_to_2019_20",\n    PRIMARY_SEASONS,\n)\n\nvalidation = build_dataset(\n    "validation_2008_09_to_2012_13",\n    VALIDATION_SEASONS,\n)\n\noverlap = set(primary["game_ids"]) & set(validation["game_ids"])\n\nif overlap:\n    raise RuntimeError(\n        f"Primary and validation datasets share {len(overlap)} games."\n    )\n\nprint(\n    "\\nDataset independence check PASSED: "\n    "the two five-season match sets are disjoint."\n)\n\n# ================================================================\n# FOUR COHORTS\n# Outfield is the main comparison; all-player is sensitivity.\n# ================================================================\n\nexperiment_specs = [\n    (\n        "primary_outfield",\n        primary,\n        primary["outfield_ids"],\n        primary["outfield_cohort"],\n        "Primary 2015/16–2019/20 — outfield only",\n        SEED + 101,\n    ),\n    (\n        "primary_all",\n        primary,\n        primary["all_ids"],\n        primary["all_cohort"],\n        "Primary 2015/16–2019/20 — all players",\n        SEED + 102,\n    ),\n    (\n        "validation_outfield",\n        validation,\n        validation["outfield_ids"],\n        validation["outfield_cohort"],\n        "Validation 2008/09–2012/13 — outfield only",\n        SEED + 103,\n    ),\n    (\n        "validation_all",\n        validation,\n        validation["all_ids"],\n        validation["all_cohort"],\n        "Validation 2008/09–2012/13 — all players",\n        SEED + 104,\n    ),\n]\n\n# ================================================================\n# PART A — FULL-DATA STATIC PCS/RAR LEADERBOARDS\n# ================================================================\n\nall_static = []\nall_pairs = []\nall_fits = []\n\nfor key, data, ids, cohort, label, seed in experiment_specs:\n    print("\\n" + "=" * 88)\n    print("STATIC PCS/RAR:", label)\n    print("=" * 88)\n\n    leaderboard, pairs, fit = static_consensus_leaderboard(\n        data["player_match"],\n        cohort,\n        ids,\n        key,\n        B=FULL_BOOTSTRAP_B,\n        seed=seed,\n    )\n\n    leaderboard["label"] = label\n    pairs["label"] = label\n    fit["label"] = label\n\n    print(\n        leaderboard[\n            [\n                "player_name",\n                "vaep90",\n                "raw_VAEP_rank",\n                "PCS",\n                "RAR",\n                "supported_rank_low",\n                "supported_rank_high",\n                "pairwise_resolution_rate",\n            ]\n        ]\n        .round(4)\n        .to_string(index=False)\n    )\n\n    all_static.append(leaderboard)\n    all_pairs.append(pairs)\n    all_fits.append(fit)\n\nstatic_all = pd.concat(all_static, ignore_index=True)\npairs_all = pd.concat(all_pairs, ignore_index=True)\nfits_all = pd.concat(all_fits, ignore_index=True)\n\nstatic_all.to_csv(\n    RAR_OUT_ROOT / "01_STATIC_PCS_RAR_leaderboards.csv",\n    index=False,\n)\n\npairs_all.to_csv(\n    RAR_OUT_ROOT / "02_STATIC_pairwise_probabilities.csv",\n    index=False,\n)\n\nfits_all.to_csv(\n    RAR_OUT_ROOT / "03_STATIC_BT_fit_summary.csv",\n    index=False,\n)\n\n# ================================================================\n# PART B — CRITICAL TEST: RAW VAEP VS RAR REPRODUCIBILITY\n# ================================================================\n\nsummaries = []\nreplicates = []\noverall_rows = []\n\nfor key, data, ids, cohort, label, seed in experiment_specs:\n    print("\\n" + "=" * 88)\n    print("INDEPENDENT REPRODUCIBILITY:", label)\n    print("=" * 88)\n\n    summary, raw, overall = independent_rar_validation(\n        data["player_match"],\n        ids,\n        key,\n        reps=VALIDATION_REPS,\n        inner_B=INNER_BOOTSTRAP_B,\n        seed=seed + 1000,\n    )\n\n    summary["label"] = label\n    raw["label"] = label\n    overall["label"] = label\n\n    print(\n        summary[\n            [\n                "n_matches_per_sample",\n                "median_raw_VAEP_spearman",\n                "median_RAR_spearman",\n                "median_RAR_minus_raw",\n                "fraction_RAR_better",\n                "fraction_equal",\n                "fraction_RAR_worse",\n            ]\n        ]\n        .round(4)\n        .to_string(index=False)\n    )\n\n    summaries.append(summary)\n    replicates.append(raw)\n    overall_rows.append(overall)\n\nsummary_all = pd.concat(summaries, ignore_index=True)\nraw_all = pd.concat(replicates, ignore_index=True)\noverall_all = pd.concat(overall_rows, ignore_index=True)\n\nsummary_all.to_csv(\n    RAR_OUT_ROOT / "04_RAR_vs_raw_reproducibility_by_n.csv",\n    index=False,\n)\n\nraw_all.to_csv(\n    RAR_OUT_ROOT / "05_RAR_vs_raw_reproducibility_replicates.csv",\n    index=False,\n)\n\noverall_all.to_csv(\n    RAR_OUT_ROOT / "06_RAR_vs_raw_overall_summary.csv",\n    index=False,\n)\n\nprint("\\n" + "=" * 88)\nprint("OVERALL RAR VALIDATION SUMMARY")\nprint("=" * 88)\nprint(\n    overall_all[\n        [\n            "label",\n            "players",\n            "maximum_testable_n",\n            "tested_n_points",\n            "n_points_RAR_higher_median",\n            "n_points_equal_median",\n            "n_points_RAR_lower_median",\n            "pooled_median_RAR_minus_raw",\n            "pooled_mean_RAR_minus_raw",\n            "pooled_fraction_RAR_better",\n            "pooled_fraction_equal",\n            "pooled_fraction_RAR_worse",\n        ]\n    ]\n    .round(4)\n    .to_string(index=False)\n)\n\n# ================================================================\n# FIGURES\n# ================================================================\n\nfor key, data, ids, cohort, label, seed in experiment_specs:\n    q = summary_all[summary_all["experiment"] == key].copy()\n\n    fig, ax = plt.subplots(figsize=(8, 5.5))\n    ax.plot(\n        q["n_matches_per_sample"],\n        q["median_raw_VAEP_spearman"],\n        marker="o",\n        label="Raw VAEP rank",\n    )\n    ax.plot(\n        q["n_matches_per_sample"],\n        q["median_RAR_spearman"],\n        marker="o",\n        label="RAR",\n    )\n    ax.axhline(\n        0.80,\n        linestyle="--",\n        label="0.80 reference",\n    )\n    ax.set_xlabel("Matches per independent sample")\n    ax.set_ylabel("Median Spearman reproducibility")\n    ax.set_title(label)\n    ax.set_ylim(-1, 1)\n    ax.legend()\n    plt.tight_layout()\n    fig.savefig(\n        RAR_OUT_ROOT / f"07_{key}_raw_vs_RAR.png",\n        dpi=180,\n    )\n    plt.close(fig)\n\n# Combined delta plot.\nfig, ax = plt.subplots(figsize=(9, 6))\n\nfor key, data, ids, cohort, label, seed in experiment_specs:\n    q = summary_all[summary_all["experiment"] == key].copy()\n    ax.plot(\n        q["n_matches_per_sample"],\n        q["median_RAR_minus_raw"],\n        marker="o",\n        label=label,\n    )\n\nax.axhline(0.0, linestyle="--")\nax.set_xlabel("Matches per independent sample")\nax.set_ylabel("Median Spearman gain: RAR − raw VAEP")\nax.set_title("Does PCS/RAR improve independent ranking reproducibility?")\nax.legend(fontsize=8)\nplt.tight_layout()\n\nfig.savefig(\n    RAR_OUT_ROOT / "08_combined_RAR_minus_raw.png",\n    dpi=180,\n)\nplt.close(fig)\n\n# Raw rank vs RAR static comparison.\nfig, ax = plt.subplots(figsize=(7, 6))\n\nfor key, data, ids, cohort, label, seed in experiment_specs:\n    q = static_all[static_all["experiment"] == key]\n    ax.scatter(\n        q["raw_VAEP_rank"],\n        q["RAR"],\n        label=label,\n    )\n\nmax_rank = int(\n    max(\n        static_all["raw_VAEP_rank"].max(),\n        static_all["RAR"].max(),\n    )\n)\n\nax.plot(\n    [1, max_rank],\n    [1, max_rank],\n    linestyle="--",\n)\n\nax.set_xlabel("Raw VAEP rank")\nax.set_ylabel("Reliability-Adjusted Rank (RAR)")\nax.set_title("Full-data static raw VAEP rank vs RAR")\nax.invert_xaxis()\nax.invert_yaxis()\nax.legend(fontsize=8)\nplt.tight_layout()\n\nfig.savefig(\n    RAR_OUT_ROOT / "09_static_raw_rank_vs_RAR.png",\n    dpi=180,\n)\nplt.close(fig)\n\n# ================================================================\n# PAPER-FACING DECISION SUMMARY\n# ================================================================\n\nlines = []\nlines.append("ReliableVAEP PCS/RAR standalone validation")\nlines.append("=" * 65)\nlines.append("")\nlines.append(\n    "PCS/RAR is supported as a reliability-adjusted static ranking "\n    "only if it is more reproducible than raw VAEP across independent "\n    "non-overlapping match samples, especially in both outfield cohorts."\n)\nlines.append("")\n\nfor _, row in overall_all.iterrows():\n    lines.append(str(row["label"]))\n    lines.append(\n        f"  n-points with higher median RAR reproducibility: "\n        f"{int(row[\'n_points_RAR_higher_median\'])}/{int(row[\'tested_n_points\'])}"\n    )\n    lines.append(\n        f"  pooled median Spearman gain (RAR - raw): "\n        f"{row[\'pooled_median_RAR_minus_raw\']:.4f}"\n    )\n    lines.append(\n        f"  replicate fraction RAR better/equal/worse: "\n        f"{row[\'pooled_fraction_RAR_better\']:.3f} / "\n        f"{row[\'pooled_fraction_equal\']:.3f} / "\n        f"{row[\'pooled_fraction_RAR_worse\']:.3f}"\n    )\n    lines.append("")\n\nlines.append("Interpretation:")\nlines.append(\n    "- If RAR is consistently better in primary and validation outfield cohorts, "\n    "describe it as a reliability-adjusted consensus ranking."\n)\nlines.append(\n    "- If improvement is inconsistent or absent, do NOT call RAR a correction "\n    "or a more accurate ranking. Keep it as a descriptive consensus summary."\n)\nlines.append(\n    "- Original VAEP/90 and supported rank bands remain visible in all cases."\n)\n\npaper_summary = "\\n".join(lines)\n\nwith open(\n    RAR_OUT_ROOT / "10_paper_facing_decision_summary.txt",\n    "w",\n    encoding="utf-8",\n) as f:\n    f.write(paper_summary)\n\nprint("\\n" + paper_summary)\n\n# Copy key canonical diagnostics into RAR output folder.\nfor dataset_key, data in [\n    ("primary", primary),\n    ("validation", validation),\n]:\n    data["cv_metrics"].to_csv(\n        RAR_OUT_ROOT / f"11_{dataset_key}_canonical_VAEP_model_metrics.csv",\n        index=False,\n    )\n    data["all_cohort"].to_csv(\n        RAR_OUT_ROOT / f"12_{dataset_key}_all_player_cohort.csv",\n        index=False,\n    )\n    data["outfield_cohort"].to_csv(\n        RAR_OUT_ROOT / f"13_{dataset_key}_outfield_cohort.csv",\n        index=False,\n    )\n\n# ZIP all PCS/RAR outputs.\nzip_base = ROOT / "ReliableVAEP_PCS_RAR_STANDALONE_RESULTS"\nzip_path = shutil.make_archive(\n    str(zip_base),\n    "zip",\n    root_dir=RAR_OUT_ROOT,\n)\n\nprint("\\n" + "=" * 88)\nprint("PCS/RAR ANALYSIS COMPLETE")\nprint("=" * 88)\nprint("Output folder:", RAR_OUT_ROOT)\nprint("Results ZIP:", zip_path)\n'

SCRIPT_PATH.write_text(
    ANALYSIS_SCRIPT,
    encoding="utf-8",
)

print("Standalone PCS/RAR analysis written to:", SCRIPT_PATH)
print("Script size:", SCRIPT_PATH.stat().st_size, "bytes")


Standalone PCS/RAR analysis written to: /kaggle/working/reliablevaep_pcs_rar_standalone.py
Script size: 61986 bytes


In [4]:

# ================================================================
# CELL 3 — RUN THE COMPLETE STANDALONE PCS/RAR ANALYSIS
# ================================================================
#
# First run can be long because canonical VAEP must be built for two
# five-season periods. If the canonical cache already exists from the
# stability notebook in the same Kaggle session, it is reused.
# ================================================================

import os
import subprocess
from pathlib import Path

BASE_DIR = Path("/kaggle/working")
ENV_PY = BASE_DIR / "rv_vaep_env" / "bin" / "python"
SCRIPT_PATH = BASE_DIR / "reliablevaep_pcs_rar_standalone.py"

if not ENV_PY.exists():
    raise RuntimeError(
        "Run Cell 1 first. The isolated Kaggle VAEP environment does not exist."
    )

if not SCRIPT_PATH.exists():
    raise RuntimeError(
        "Run Cell 2 first. The standalone analysis script does not exist."
    )

child_env = os.environ.copy()
child_env["MPLBACKEND"] = "Agg"
child_env.pop("MATPLOTLIBRC", None)

print("Running:", SCRIPT_PATH)
print("Using isolated Python:", ENV_PY)
print()

process = subprocess.Popen(
    [str(ENV_PY), "-u", str(SCRIPT_PATH)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=child_env,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"PCS/RAR analysis exited with code {return_code}. "
        "The exact traceback is shown immediately above."
    )

print("\nStandalone PCS/RAR analysis finished successfully.")


Running: /kaggle/working/reliablevaep_pcs_rar_standalone.py
Using isolated Python: /kaggle/working/rv_vaep_env/bin/python

ReliableVAEP canonical dual-period validation
Primary: 2015/16, 2016/17, 2017/18, 2018/19, 2019/20
Validation: 2008/09, 2009/10, 2010/11, 2011/12, 2012/13
Outputs: /kaggle/working/rv_dual_period_canonical_outputs


BUILDING: primary_2015_16_to_2019_20

Barcelona match availability:
season
2015/16    38
2016/17    34
2017/18    36
2018/19    34
2019/20    33
Total matches: 175

primary_2015_16_to_2019_20: SPADL: 100%|██████████| 175/175 [05:27<00:00,  1.87s/match]

primary_2015_16_to_2019_20: features: 100%|██████████| 175/175 [00:34<00:00,  5.05match/s]
Feature rows=388,562, features=568, score rate=0.0131, concede rate=0.0022

Cross-fit model means:
fit_matches                 112.0000
calibration_matches          28.0000
test_matches                 35.0000
scores_auc                    0.8158
scores_ap                     0.2414
scores_brier_baseline         0.0

In [5]:

# ================================================================
# CELL 4 — PRINT THE FINAL DECISION TABLES
# ================================================================

from pathlib import Path
import csv

ROOT = Path("/kaggle/working")
OUT = ROOT / "rv_pcs_rar_outputs"

def print_csv(path):
    if not path.exists():
        print("Missing:", path)
        return

    print("\n" + "=" * 110)
    print(path.name)
    print("=" * 110)

    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.reader(f))

    if not rows:
        print("(empty)")
        return

    widths = [
        min(
            40,
            max(
                len(str(row[i])) if i < len(row) else 0
                for row in rows
            ),
        )
        for i in range(len(rows[0]))
    ]

    for row in rows:
        vals = []
        for i, value in enumerate(row):
            value = str(value)
            if len(value) > widths[i]:
                value = value[: widths[i] - 3] + "..."
            vals.append(value.ljust(widths[i]))
        print(" | ".join(vals))

print_csv(OUT / "06_RAR_vs_raw_overall_summary.csv")
print_csv(OUT / "04_RAR_vs_raw_reproducibility_by_n.csv")
print_csv(OUT / "01_STATIC_PCS_RAR_leaderboards.csv")

summary_txt = OUT / "10_paper_facing_decision_summary.txt"
if summary_txt.exists():
    print("\n" + "=" * 110)
    print("PAPER-FACING DECISION SUMMARY")
    print("=" * 110)
    print(summary_txt.read_text(encoding="utf-8"))

print("\nResults folder:")
print(OUT)

print("\nZIP:")
print(ROOT / "ReliableVAEP_PCS_RAR_STANDALONE_RESULTS.zip")



06_RAR_vs_raw_overall_summary.csv
experiment          | players | minimum_matches_in_cohort | maximum_testable_n | tested_n_points | validation_reps_per_n | inner_bootstrap_draws | n_points_RAR_higher_median | n_points_equal_median | n_points_RAR_lower_median | pooled_median_RAR_minus_raw | pooled_mean_RAR_minus_raw | pooled_fraction_RAR_better | pooled_fraction_equal | pooled_fraction_RAR_worse | label                                   
primary_outfield    | 7       | 141                       | 70                 | 14              | 200                   | 500                   | 1                          | 7                     | 6                         | 0.0                         | -0.003364547555122291     | 0.21392857142857144        | 0.4739285714285714    | 0.31214285714285717       | Primary 2015/16–2019/20 — outfield only 
primary_all         | 8       | 137                       | 68                 | 14              | 200                   | 500                   | 3 


## Final Kaggle output

When Cell 3 finishes successfully, the complete results are saved to:

`/kaggle/working/rv_pcs_rar_outputs/`

and the downloadable ZIP is:

`/kaggle/working/ReliableVAEP_PCS_RAR_STANDALONE_RESULTS.zip`

The most important files are:

- `01_STATIC_PCS_RAR_leaderboards.csv`  
  The requested one-number static ranking: raw VAEP rank, PCS, RAR, supported rank band, and pairwise reliability.

- `04_RAR_vs_raw_reproducibility_by_n.csv`  
  Raw VAEP vs RAR reproducibility at each tested match count.

- `05_RAR_vs_raw_reproducibility_replicates.csv`  
  Every repeated independent match-split result.

- `06_RAR_vs_raw_overall_summary.csv`  
  The primary decision table showing whether RAR actually improved reproducibility.

- `10_paper_facing_decision_summary.txt`  
  Conservative interpretation text generated directly from the results.

Do not call RAR a "corrected rank" or "more accurate rank" unless the independent reproducibility results actually support that claim.


In [ ]:

from pathlib import Path

zip_path = Path(
    "/kaggle/working/ReliableVAEP_PCS_RAR_STANDALONE_RESULTS.zip"
)

if zip_path.exists():
    print("Results ZIP is ready:")
    print(zip_path)
else:
    print(
        "Results ZIP does not exist yet. "
        "Run the complete analysis first."
    )
